# 🫁 Pneumonia Detection with Vision Transformers — Statistical Rigour Edition

**MSc Dissertation COM726 · Rahul Gupta · Southampton Solent University**

A Vision Transformer (ViT-Base/16, ImageNet-pretrained) is trained on the
[Kaggle Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)
dataset. The notebook then adds a **paired statistical comparison layer**, so the
reported differences between architectures can be defended as significant rather
than incidental.

| Analysis | Question it answers |
|---|---|
| Wilson score 95% CIs | How uncertain are accuracy, sensitivity and specificity? |
| Exact McNemar test + effect sizes | Is the paired accuracy difference real? |
| DeLong paired AUC test | Is the ROC-AUC difference real? |
| Paired bootstrap (10,000 resamples) | Distribution-free CI for the accuracy/AUC difference |
| Holm–Bonferroni correction | Does the conclusion survive multiple comparisons? |

**New artefacts:** `test_predictions.csv` (per-image predictions) and
`statistical_results.json` (every statistic, ready for the dissertation appendix).

**Kaggle setup:** attach the dataset *Chest X-Ray Images (Pneumonia)* and enable a
GPU accelerator (T4/P100), then **Run → Run All**. Training dominates the runtime
(~30–40 minutes); the statistical analyses take a few seconds.

**Adding baselines.** If ResNet-50 or VGG16 are evaluated on the *same cleaned test
set*, export their per-image predictions with the columns
`image_path, y_true, y_pred, prob_normal, prob_pneumonia` and place the CSV in
`/kaggle/working` or attach it as a Kaggle dataset. Step 6.2 discovers files named
`*predictions*.csv` automatically and skips any file whose test set does not match.

> **Medical disclaimer.** Research and educational use only — not a clinical
> diagnostic tool.


In [ ]:
import os
import gc
import glob
import json
import warnings
import time
warnings.filterwarnings('ignore')

# =============================================================================
# STEP 0: IMPORTS AND KAGGLE SETUP
# =============================================================================

print("🚀 KAGGLE PNEUMONIA DETECTION PIPELINE STARTING...")
print("=" * 70)

# Core libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm

# Data science libraries
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Machine learning
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                           roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats

# Kaggle environment check
print("🔍 Checking Kaggle environment...")
try:
    # Check if we're in Kaggle environment by looking for common Kaggle paths
    if os.path.exists('/kaggle/input') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        print("✅ Kaggle environment detected")
    else:
        print("⚠️  Running outside Kaggle - paths may need adjustment")
except Exception:
    print("⚠️  Environment check skipped")

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🎯 Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")


def free_gpu_cache():
    """Release cached GPU memory. Safe to call on CPU-only machines."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# KAGGLE-OPTIMIZED CONFIGURATION
print("⚙️ Loading Kaggle-optimized configuration...")

# Dataset path - Kaggle mounts the same dataset at different folder depths
# depending on how it was attached, so resolve the root automatically.
DATA_PATH_CANDIDATES = [
    "/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray",
    "/kaggle/input/chest-xray-pneumonia/chest_xray",
    "/kaggle/input/chest-xray-pneumonia",
]


def is_dataset_root(path):
    """True when path contains train/NORMAL and train/PNEUMONIA."""
    return all(os.path.isdir(os.path.join(path, "train", name))
               for name in ("NORMAL", "PNEUMONIA"))


DATA_PATH = next((candidate for candidate in DATA_PATH_CANDIDATES
                  if is_dataset_root(candidate)), None)

if DATA_PATH is None:            # fall back to a search (renamed dataset slug)
    for match in glob.glob("/kaggle/input/**/train/NORMAL", recursive=True):
        candidate = os.path.dirname(os.path.dirname(match))
        if is_dataset_root(candidate):
            DATA_PATH = candidate
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Chest X-ray dataset not found under /kaggle/input. Attach the Kaggle "
        "dataset 'Chest X-Ray Images (Pneumonia)' and confirm that it contains "
        "train/NORMAL and train/PNEUMONIA.")

DATA_PATH = os.path.abspath(DATA_PATH)
print(f"📂 Dataset root resolved: {DATA_PATH}")

# Training configuration - optimized for Kaggle Tesla T4
BATCH_SIZE = 32                    # Good balance for Tesla T4
NUM_EPOCHS = 10                    # Standard training duration
LEARNING_RATE = 1e-4              # Optimal for ViT fine-tuning
WEIGHT_DECAY = 1e-4               # Regularization
NUM_WORKERS = 0                   # Fix multiprocessing error - KAGGLE SAFE

# Quality control settings
OUTLIER_THRESHOLD = 2.5           # Moderate strictness
REMOVE_PROBLEMATIC = True         # Enable quality filtering

# Model settings
MODEL_NAME = 'vit_base_patch16_224'
NUM_CLASSES = 2
INPUT_SIZE = 224

print("✅ Configuration loaded:")
print(f"   Dataset: {DATA_PATH}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Quality threshold: {OUTLIER_THRESHOLD}")

In [ ]:
print("\n" + "="*70)
print("📊 STEP 1: DATASET EXPLORATION AND COUNTING")
print("="*70)

def count_and_visualize_dataset(data_path):
    """
    Comprehensive dataset analysis with beautiful visualizations
    """
    print("🔍 Analyzing dataset structure...")
    
    data_stats = {}
    splits = ['train', 'val', 'test']
    classes = ['NORMAL', 'PNEUMONIA']
    
    # Count images in each category
    for split in splits:
        data_stats[split] = {}
        split_path = os.path.join(data_path, split)
        
        if not os.path.exists(split_path):
            print(f"⚠️ Warning: {split_path} not found!")
            continue
            
        total_split = 0
        for class_name in classes:
            class_path = os.path.join(split_path, class_name)
            if os.path.exists(class_path):
                files = [f for f in os.listdir(class_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                count = len(files)
                data_stats[split][class_name] = count
                total_split += count
            else:
                data_stats[split][class_name] = 0
                
        data_stats[split]['total'] = total_split
    
    # Create comprehensive statistics
    rows = []
    for split in splits:
        if split in data_stats:
            for class_name in classes:
                rows.append({
                    'Split': split.upper(),
                    'Class': class_name,
                    'Count': data_stats[split][class_name],
                    'Total_in_Split': data_stats[split]['total']
                })

    df_stats = pd.DataFrame(rows)
    df_stats['Percentage'] = (df_stats['Count'] / df_stats['Total_in_Split'] * 100).round(1)
    
    # Print statistics
    pivot_count = df_stats.pivot(index='Split', columns='Class', values='Count')
    pivot_pct = df_stats.pivot(index='Split', columns='Class', values='Percentage')
    
    print("\n📈 DATASET STATISTICS:")
    print("Sample Counts:")
    print(pivot_count)
    print("\nPercentages within splits:")
    print(pivot_pct)
    
    # Overall summary
    total_normal = df_stats[df_stats['Class'] == 'NORMAL']['Count'].sum()
    total_pneumonia = df_stats[df_stats['Class'] == 'PNEUMONIA']['Count'].sum()
    total_all = total_normal + total_pneumonia
    
    print(f"\n🎯 OVERALL SUMMARY:")
    print(f"   Total Images: {total_all:,}")
    print(f"   NORMAL: {total_normal:,} ({total_normal/total_all*100:.1f}%)")
    print(f"   PNEUMONIA: {total_pneumonia:,} ({total_pneumonia/total_all*100:.1f}%)")
    print(f"   Imbalance Ratio: 1:{total_pneumonia/total_normal:.2f}")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('📊 Chest X-Ray Dataset Analysis', fontsize=16, fontweight='bold')
    
    # 1. Count by split
    ax1 = axes[0, 0]
    pivot_count.plot(kind='bar', ax=ax1, color=['skyblue', 'lightcoral'])
    ax1.set_title('Sample Count by Split')
    ax1.set_xlabel('Split')
    ax1.set_ylabel('Count')
    ax1.legend(title='Class')
    
    # 2. Overall distribution
    ax2 = axes[0, 1]
    ax2.pie([total_normal, total_pneumonia], labels=['NORMAL', 'PNEUMONIA'], 
            autopct='%1.1f%%', colors=['skyblue', 'lightcoral'])
    ax2.set_title('Overall Class Distribution')
    
    # 3. Percentage by split
    ax3 = axes[1, 0]
    pivot_pct.plot(kind='bar', ax=ax3, color=['skyblue', 'lightcoral'])
    ax3.set_title('Class Percentage by Split')
    ax3.set_xlabel('Split')
    ax3.set_ylabel('Percentage (%)')
    ax3.legend(title='Class')
    
    # 4. Heatmap
    ax4 = axes[1, 1]
    sns.heatmap(pivot_count, annot=True, fmt='d', cmap='YlOrRd', ax=ax4)
    ax4.set_title('Count Heatmap')
    
    plt.tight_layout()
    plt.show()
    
    return data_stats, total_all, total_normal, total_pneumonia

# Run dataset exploration
data_stats, total_images, total_normal, total_pneumonia = count_and_visualize_dataset(DATA_PATH)

In [ ]:
print("\n" + "="*70)
print("🔬 STEP 2: ENHANCED MEDICAL-GRADE QUALITY CONTROL")
print("="*70)

def calculate_medical_quality_metrics(image_path):
    """
    Calculate comprehensive medical image quality metrics
    """
    try:
        # Load and verify image
        pil_img = Image.open(image_path).convert('RGB')
        cv_img = cv2.imread(image_path)
        if cv_img is None:
            return None
            
        cv_gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
        width, height = pil_img.size
        
        # Verify image integrity
        pil_img.verify()
        pil_img = Image.open(image_path).convert('RGB')  # Reload
        
        # 1. Brightness (exposure level)
        brightness = np.mean(cv_gray)
        
        # 2. Contrast (image clarity)
        contrast = np.std(cv_gray)
        
        # 3. Sharpness (focus quality)
        laplacian = cv2.Laplacian(cv_gray, cv2.CV_64F)
        sharpness = laplacian.var()
        
        # 4. Noise level
        blurred = cv2.GaussianBlur(cv_gray, (5, 5), 0)
        noise_level = np.std(cv_gray - blurred)
        
        # 5. Dynamic range
        dynamic_range = np.max(cv_gray) - np.min(cv_gray)
        
        # 6. Exposure analysis
        overexposed = np.sum(cv_gray > 240) / (width * height)
        underexposed = np.sum(cv_gray < 15) / (width * height)
        
        # 7. Detail level (edge density)
        edges = cv2.Canny(cv_gray, 50, 150)
        edge_density = np.sum(edges) / (width * height)
        
        # 8. Signal-to-noise ratio
        signal = np.mean(cv_gray)
        noise = np.std(cv_gray)
        snr = signal / noise if noise > 0 else 0
        
        return {
            'image_path': image_path,
            'width': width,
            'height': height,
            'brightness': brightness,
            'contrast': contrast,
            'sharpness': sharpness,
            'noise_level': noise_level,
            'dynamic_range': dynamic_range,
            'overexposed_ratio': overexposed,
            'underexposed_ratio': underexposed,
            'edge_density': edge_density,
            'snr': snr,
            'is_valid': True
        }
        
    except Exception as e:
        return {
            'image_path': image_path,
            'error': str(e),
            'is_valid': False
        }

def analyze_image_quality(data_path):
    """
    Analyze quality metrics across the entire dataset
    """
    print("🔍 Analyzing image quality metrics...")
    
    all_metrics = []
    splits = ['train', 'val', 'test']
    classes = ['NORMAL', 'PNEUMONIA']
    
    for split in splits:
        split_path = os.path.join(data_path, split)
        if not os.path.exists(split_path):
            continue
            
        for class_name in classes:
            class_path = os.path.join(split_path, class_name)
            if not os.path.exists(class_path):
                continue
                
            files = [f for f in os.listdir(class_path) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            
            print(f"   📊 {split}/{class_name}: {len(files)} files")
            
            # Process files with progress bar
            for file in tqdm(files, desc=f"{split}/{class_name}"):
                file_path = os.path.join(class_path, file)
                metrics = calculate_medical_quality_metrics(file_path)
                
                if metrics:
                    metrics['split'] = split
                    metrics['class'] = class_name
                    metrics['filename'] = file
                    all_metrics.append(metrics)
    
    return pd.DataFrame(all_metrics)

def detect_quality_problems(df_metrics, threshold=2.5):
    """
    Detect problematic images using medical imaging standards
    """
    print(f"\n🎯 Detecting quality issues (threshold: {threshold})")
    
    problematic = set()
    reasons = {}
    
    valid_df = df_metrics[df_metrics['is_valid'] == True]
    if len(valid_df) == 0:
        return problematic, reasons
    
    # Statistical outlier detection
    metrics = ['brightness', 'contrast', 'sharpness', 'noise_level', 
              'dynamic_range', 'edge_density', 'snr']
    
    print("📊 Statistical outlier detection:")
    for metric in metrics:
        if metric in valid_df.columns:
            z_scores = np.abs(stats.zscore(valid_df[metric]))
            outliers = valid_df[z_scores > threshold]
            
            for idx, row in outliers.iterrows():
                img_path = row['image_path']
                problematic.add(img_path)
                if img_path not in reasons:
                    reasons[img_path] = []
                reasons[img_path].append(f"{metric}_outlier")
            
            print(f"   {metric}: {len(outliers)} outliers")
    
    # Medical imaging thresholds
    print("\n🏥 Medical imaging quality checks:")
    
    # Dark images
    dark_threshold = np.percentile(valid_df['brightness'], 5)
    dark = valid_df[valid_df['brightness'] < dark_threshold]
    
    # Bright images  
    bright_threshold = np.percentile(valid_df['brightness'], 95)
    bright = valid_df[valid_df['brightness'] > bright_threshold]
    
    # Low contrast
    contrast_threshold = np.percentile(valid_df['contrast'], 10)
    low_contrast = valid_df[valid_df['contrast'] < contrast_threshold]
    
    # Blurry images
    blur_threshold = np.percentile(valid_df['sharpness'], 10)
    blurry = valid_df[valid_df['sharpness'] < blur_threshold]
    
    # Exposure problems
    overexposed = valid_df[valid_df['overexposed_ratio'] > 0.15]
    underexposed = valid_df[valid_df['underexposed_ratio'] > 0.15]
    
    quality_checks = [
        (dark, "very_dark"), (bright, "very_bright"),
        (low_contrast, "low_contrast"), (blurry, "blurry"),
        (overexposed, "overexposed"), (underexposed, "underexposed")
    ]
    
    for issue_df, issue_type in quality_checks:
        for _, row in issue_df.iterrows():
            img_path = row['image_path']
            problematic.add(img_path)
            if img_path not in reasons:
                reasons[img_path] = []
            reasons[img_path].append(issue_type)
        print(f"   {issue_type}: {len(issue_df)} images")
    
    # Corrupted images
    corrupted = df_metrics[df_metrics['is_valid'] == False]
    for _, row in corrupted.iterrows():
        img_path = row['image_path']
        problematic.add(img_path)
        if img_path not in reasons:
            reasons[img_path] = []
        reasons[img_path].append("corrupted")
    
    print(f"   corrupted: {len(corrupted)} images")
    
    print(f"\n📊 Quality summary:")
    print(f"   Total problematic: {len(problematic)} ({len(problematic)/len(df_metrics)*100:.1f}%)")
    
    return problematic, reasons

def visualize_quality_metrics(df_metrics):
    """
    Create medical-grade quality visualizations
    """
    valid_df = df_metrics[df_metrics['is_valid'] == True]
    if len(valid_df) == 0:
        return
    
    print("📊 Creating quality distribution plots...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('🔬 Medical Image Quality Analysis', fontsize=16, fontweight='bold')
    
    metrics = [
        ('brightness', 'Brightness (Exposure)'),
        ('contrast', 'Contrast (Clarity)'),
        ('sharpness', 'Sharpness (Focus)'),
        ('noise_level', 'Noise Level'),
        ('edge_density', 'Detail Level'),
        ('snr', 'Signal-to-Noise Ratio')
    ]
    
    for i, (metric, title) in enumerate(metrics):
        row, col = i // 3, i % 3
        ax = axes[row, col]
        
        # Plot by class
        for class_name in valid_df['class'].unique():
            data = valid_df[valid_df['class'] == class_name][metric]
            ax.hist(data, alpha=0.7, bins=30, label=class_name, density=True)
        
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def create_clean_dataset(data_path, remove_problematic=True, threshold=2.5):
    """
    Create quality-controlled clean dataset
    """
    print("🧹 Creating clean dataset with quality control...")
    
    # Analyze quality
    df_metrics = analyze_image_quality(data_path)
    
    # Detect problems
    problematic, reasons = detect_quality_problems(df_metrics, threshold)
    
    # Visualize quality
    visualize_quality_metrics(df_metrics)
    
    # Create clean file lists
    clean_files = {
        'train': {'NORMAL': [], 'PNEUMONIA': []},
        'val': {'NORMAL': [], 'PNEUMONIA': []},
        'test': {'NORMAL': [], 'PNEUMONIA': []}
    }
    
    removed = 0
    valid_df = df_metrics[df_metrics['is_valid'] == True]
    
    for _, row in valid_df.iterrows():
        path = row['image_path']
        split = row['split'] 
        class_name = row['class']
        
        if remove_problematic and path in problematic:
            removed += 1
            continue
            
        clean_files[split][class_name].append(path)
    
    # Summary
    print(f"\n✅ QUALITY CONTROL COMPLETE:")
    print(f"   Total analyzed: {len(df_metrics):,}")
    print(f"   Valid images: {len(valid_df):,}")
    print(f"   Problematic: {len(problematic):,}")
    print(f"   Removed: {removed:,}")
    
    total_clean = 0
    for split in ['train', 'val', 'test']:
        normal = len(clean_files[split]['NORMAL'])
        pneumonia = len(clean_files[split]['PNEUMONIA'])
        split_total = normal + pneumonia
        total_clean += split_total
        print(f"   {split.upper()}: {split_total:,} (NORMAL: {normal:,}, PNEUMONIA: {pneumonia:,})")
    
    print(f"   FINAL CLEAN DATASET: {total_clean:,} images")
    
    return clean_files, df_metrics

# Run enhanced quality control
clean_files, quality_df = create_clean_dataset(
    DATA_PATH, 
    remove_problematic=REMOVE_PROBLEMATIC,
    threshold=OUTLIER_THRESHOLD
)

In [ ]:
print("\n" + "="*70)
print("🔄 STEP 3: CUSTOM DATASET WITH ENHANCED TRANSFORMS")
print("="*70)

class CleanMedicalDataset(Dataset):
    """
    Custom dataset for clean medical images with enhanced preprocessing
    """
    def __init__(self, file_dict, transform=None):
        self.transform = transform
        self.files = []
        self.labels = []
        self.class_to_idx = {'NORMAL': 0, 'PNEUMONIA': 1}
        
        for class_name, paths in file_dict.items():
            for path in paths:
                self.files.append(path)
                self.labels.append(self.class_to_idx[class_name])
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        image = Image.open(self.files[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Enhanced medical imaging transforms
print("🎨 Setting up medical-grade transforms...")

train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Create datasets
train_dataset = CleanMedicalDataset(clean_files['train'], train_transform)
val_dataset = CleanMedicalDataset(clean_files['val'], val_transform)
test_dataset = CleanMedicalDataset(clean_files['test'], val_transform)

print(f"✅ Clean datasets created:")
print(f"   Training: {len(train_dataset):,}")
print(f"   Validation: {len(val_dataset):,}")
print(f"   Test: {len(test_dataset):,}")

# Data loaders - SAFE CONFIGURATION FOR KAGGLE
train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=False)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
test_loader = DataLoader(test_dataset, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

# Class weights for imbalanced data
labels = train_dataset.labels
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = torch.FloatTensor(class_weights)
print(f"✅ Class weights: {class_weights}")

In [ ]:
print("\n" + "="*70)
print("🤖 STEP 4: VISION TRANSFORMER TRAINING")
print("="*70)

def create_vit_model():
    """Create and configure Vision Transformer"""
    print(f"🔧 Creating {MODEL_NAME} model...")
    model = timm.create_model(MODEL_NAME, pretrained=True)
    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    
    # Initialize new head
    nn.init.xavier_uniform_(model.head.weight)
    nn.init.zeros_(model.head.bias)
    
    return model

def train_vit_model(model, train_loader, val_loader, class_weights):
    """Train ViT with comprehensive monitoring"""
    model = model.to(device)
    
    # Training setup
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    
    # Training history
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = -1.0
    best_model = None
    
    print(f"🎯 Training configuration:")
    print(f"   Model: {MODEL_NAME}")
    print(f"   Epochs: {NUM_EPOCHS}")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Learning rate: {LEARNING_RATE}")
    print(f"   Device: {device}")
    print("-" * 60)
    
    for epoch in range(NUM_EPOCHS):
        start_time = time.time()
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
        for data, targets in train_pbar:
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += targets.size(0)
            train_correct += predicted.eq(targets).sum().item()
            
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")
            for data, targets in val_pbar:
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()
                
                val_pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'Acc': f'{100.*val_correct/val_total:.2f}%'
                })
        
        # Calculate metrics
        train_loss_avg = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total
        val_loss_avg = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total
        
        # Save history
        history['train_loss'].append(train_loss_avg)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss_avg)
        history['val_acc'].append(val_acc)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            # Detach and clone every tensor: state_dict().copy() only copies the
            # dict, so later optimizer steps would silently overwrite the snapshot.
            best_model = {name: tensor.detach().clone()
                          for name, tensor in model.state_dict().items()}
            print(f"🎉 New best validation accuracy: {val_acc:.2f}%")
        
        # Print epoch results
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({epoch_time:.1f}s):")
        print(f"  Train: Loss={train_loss_avg:.4f}, Acc={train_acc:.2f}%")
        print(f"  Val:   Loss={val_loss_avg:.4f}, Acc={val_acc:.2f}%")
        print(f"  LR: {scheduler.get_last_lr()[0]:.6f}")
        print("-" * 60)
        
        scheduler.step()
        free_gpu_cache()
        gc.collect()
    
    # Load best model
    model.load_state_dict(best_model)
    print(f"✅ Training complete! Best validation accuracy: {best_val_acc:.2f}%")
    
    return model, history

# Create and train model
model = create_vit_model()
print(f"✅ Model created: {sum(p.numel() for p in model.parameters()):,} parameters")

trained_model, training_history = train_vit_model(model, train_loader, val_loader, class_weights)

In [ ]:
print("\n" + "="*70)
print("🎯 STEP 5: COMPREHENSIVE EVALUATION")
print("="*70)

def evaluate_model_comprehensive(model, test_loader):
    """Comprehensive model evaluation with visualizations"""
    model.eval()
    predictions = []
    targets = []
    probabilities = []
    correct = 0
    total = 0
    
    print("🔍 Evaluating on test set...")
    
    with torch.no_grad():
        for data, target in tqdm(test_loader, desc="Testing"):
            data, target = data.to(device), target.to(device)
            output = model(data)
            prob = F.softmax(output, dim=1)
            pred = output.argmax(dim=1)
            
            predictions.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())
            probabilities.extend(prob.cpu().numpy())
            
            total += target.size(0)
            correct += pred.eq(target).sum().item()
    
    accuracy = 100. * correct / total
    
    print(f"\n✅ TEST RESULTS:")
    print(f"   Accuracy: {accuracy:.2f}%")
    print(f"   Samples: {total:,}")
    print(f"   Correct: {correct:,}")
    
    # Detailed metrics
    class_names = ['NORMAL', 'PNEUMONIA']
    print(f"\n📊 CLASSIFICATION REPORT:")
    print(classification_report(targets, predictions, target_names=class_names))
    
    # Confusion matrix
    cm = confusion_matrix(targets, predictions)
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('🎯 Model Evaluation Results', fontsize=16, fontweight='bold')
    
    # 1. Confusion Matrix
    ax1 = axes[0, 0]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
                xticklabels=class_names, yticklabels=class_names)
    ax1.set_title('Confusion Matrix')
    ax1.set_xlabel('Predicted')
    ax1.set_ylabel('Actual')
    
    # 2. Training curves
    ax2 = axes[0, 1]
    epochs = range(1, len(training_history['train_loss']) + 1)
    ax2.plot(epochs, training_history['train_loss'], 'b-', label='Train Loss')
    ax2.plot(epochs, training_history['val_loss'], 'r-', label='Val Loss')
    ax2.set_title('Training Loss')
    ax2.legend()
    ax2.grid(True)
    
    # 3. Accuracy curves
    ax3 = axes[1, 0]
    ax3.plot(epochs, training_history['train_acc'], 'b-', label='Train Acc')
    ax3.plot(epochs, training_history['val_acc'], 'r-', label='Val Acc')
    ax3.set_title('Training Accuracy')
    ax3.legend()
    ax3.grid(True)
    
    # 4. ROC Curve
    ax4 = axes[1, 1]
    prob_pos = [p[1] for p in probabilities]
    fpr, tpr, _ = roc_curve(targets, prob_pos)
    auc = roc_auc_score(targets, prob_pos)
    ax4.plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})')
    ax4.plot([0, 1], [0, 1], 'k--')
    ax4.set_title('ROC Curve')
    ax4.legend()
    ax4.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'accuracy': accuracy,
        'predictions': predictions,
        'targets': targets,
        'probabilities': probabilities,
        'auc': auc
    }

# Evaluate model
results = evaluate_model_comprehensive(trained_model, test_loader)

## Step 6 — Statistical rigour: is the ViT actually better?

A single accuracy figure cannot support a claim of architectural superiority: the
test set is finite (n = 380 after quality control), so differences of a few images
are indistinguishable from noise. This section therefore treats the comparison as a
**paired** problem — every model is scored on exactly the same images — and reports
effect sizes with confidence intervals alongside p-values.

### Methods

1. **Wilson score intervals** (Wilson, 1927; Newcombe, 1998) for accuracy,
   sensitivity and specificity. Unlike the Wald interval, coverage stays close to
   nominal when proportions approach 0 or 1, or when n is small.
2. **Exact McNemar test** on the 2×2 table of paired correctness (Dietterich,
   1998). Only discordant pairs carry information. The exact two-sided binomial
   p-value is primary; the Edwards continuity-corrected χ² statistic is reported
   for comparison. Effect sizes: the risk difference (accuracy difference in
   percentage points, with a Wald 95% CI) and the conditional discordant-pairs
   odds ratio with an exact Clopper–Pearson interval.
3. **DeLong test** for two correlated ROC AUCs (DeLong, DeLong & Clarke-Pearson,
   1988), computed with the fast mid-rank algorithm of Sun & Xu (2014). This is
   the standard test for comparing AUCs calculated on the same cases.
4. **Paired percentile bootstrap** (Efron & Tibshirani, 1993): 10,000 resamples of
   the test images, with both models scored on each resample, giving a
   distribution-free CI for the accuracy and AUC differences.
5. **Holm–Bonferroni step-down correction** (Holm, 1979) when the ViT is compared
   with more than one baseline, controlling the family-wise error rate at 5%.

### Reading the output

* A difference is claimed only when the **Holm-adjusted** p-value is below 0.05.
* `OR` is the odds of the ViT being correct on a discordant pair; `OR > 1` favours
  the ViT.
* `Δacc` is ViT accuracy minus baseline accuracy, in percentage points.
* The bootstrap p-value is approximate and is reported for completeness; McNemar
  and DeLong are the confirmatory tests.

### References

DeLong, E.R., DeLong, D.M. & Clarke-Pearson, D.L. (1988) 'Comparing the areas
under two or more correlated receiver operating characteristic curves: a
nonparametric approach', *Biometrics*, 44(3), pp. 837–845.

Dietterich, T.G. (1998) 'Approximate statistical tests for comparing supervised
classification learning algorithms', *Neural Computation*, 10(7), pp. 1895–1923.

Efron, B. & Tibshirani, R.J. (1993) *An Introduction to the Bootstrap*. New York:
Chapman & Hall.

Holm, S. (1979) 'A simple sequentially rejective multiple test procedure',
*Scandinavian Journal of Statistics*, 6(2), pp. 65–70.

Newcombe, R.G. (1998) 'Two-sided confidence intervals for the single proportion:
comparison of seven methods', *Statistics in Medicine*, 17(8), pp. 857–872.

Sun, X. & Xu, W. (2014) 'Fast implementation of DeLong's algorithm for comparing
the areas under correlated receiver operating characteristic curves', *IEEE Signal
Processing Letters*, 21(11), pp. 1389–1393.

### Methodological note

Quality-control thresholds in Step 2 are estimated across all splits. For a strictly
leakage-free protocol they should be fitted on the training split alone and then
applied unchanged to validation and test. This does not affect the paired tests
below (which condition on the fixed test set), but it is worth stating as a
limitation.


In [ ]:
# =============================================================================
# STEP 6: STATISTICAL RIGOUR — TOOLKIT
# =============================================================================
# Pure functions implementing the paired tests used in Steps 6.1 and 6.2.
# Dependencies: numpy, pandas, scipy, scikit-learn (all available on Kaggle).

STAT_ALPHA = 0.05                 # two-sided significance level
N_BOOTSTRAP = 10_000              # paired bootstrap resamples
BOOTSTRAP_SEED = 42
Z_95 = 1.959963984540054          # standard-normal 97.5th percentile


# --- Proportions -------------------------------------------------------------
def wilson_interval(successes, n, z=Z_95):
    """Wilson score interval for a binomial proportion (Newcombe, 1998)."""
    if n <= 0:
        raise ValueError("n must be positive")
    p = successes / n
    denominator = 1.0 + z * z / n
    centre = (p + z * z / (2.0 * n)) / denominator
    half_width = z * np.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n)) / denominator
    return max(0.0, centre - half_width), min(1.0, centre + half_width)


def proportion_summary(successes, n):
    """Point estimate plus a 95% Wilson interval, packaged for tables and JSON."""
    low, high = wilson_interval(successes, n)
    return {
        "estimate": successes / n,
        "ci_low": float(low),
        "ci_high": float(high),
        "successes": int(successes),
        "n": int(n),
    }


def clopper_pearson(successes, n, alpha=STAT_ALPHA):
    """Exact (Clopper-Pearson) binomial confidence interval."""
    low = 0.0 if successes == 0 else float(
        stats.beta.ppf(alpha / 2.0, successes, n - successes + 1))
    high = 1.0 if successes == n else float(
        stats.beta.ppf(1.0 - alpha / 2.0, successes + 1, n - successes))
    return low, high


def _odds(share):
    """Convert a proportion to odds, guarding the boundaries."""
    if share <= 0.0:
        return 0.0
    if share >= 1.0:
        return float("inf")
    return share / (1.0 - share)


# --- Confusion matrix and per-model summaries -------------------------------
def confusion_counts(y_true, y_pred):
    """TN/FP/FN/TP with label 0 = NORMAL and label 1 = PNEUMONIA."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    return {
        "tn": int(np.sum((y_true == 0) & (y_pred == 0))),
        "fp": int(np.sum((y_true == 0) & (y_pred == 1))),
        "fn": int(np.sum((y_true == 1) & (y_pred == 0))),
        "tp": int(np.sum((y_true == 1) & (y_pred == 1))),
    }


def model_summary(y_true, y_pred):
    """Accuracy, sensitivity and specificity as proportions with Wilson CIs."""
    counts = confusion_counts(y_true, y_pred)
    n = int(len(y_true))
    summary = {
        "n": n,
        "confusion": counts,
        "accuracy": proportion_summary(counts["tn"] + counts["tp"], n),
    }
    positives = counts["tp"] + counts["fn"]
    negatives = counts["tn"] + counts["fp"]
    if positives:
        summary["sensitivity"] = proportion_summary(counts["tp"], positives)
    if negatives:
        summary["specificity"] = proportion_summary(counts["tn"], negatives)
    return summary


# --- AUC and DeLong ---------------------------------------------------------
def auc_midrank(y_true, scores):
    """Rank-based (Mann-Whitney) AUC; equals roc_auc_score, ties included."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    positive = y_true == 1
    m = int(positive.sum())
    n = int((~positive).sum())
    if m == 0 or n == 0:
        raise ValueError("AUC requires both classes in y_true")
    ranks = stats.rankdata(scores, method="average")
    return float((ranks[positive].sum() - m * (m + 1) / 2.0) / (m * n))


def _midrank(values):
    """Mid-ranks (ties averaged) of a one-dimensional array."""
    values = np.asarray(values, dtype=float)
    order = np.argsort(values, kind="mergesort")
    ordered = values[order]
    ranks = np.empty(len(values), dtype=float)
    start = 0
    while start < len(values):
        end = start
        while end < len(values) and ordered[end] == ordered[start]:
            end += 1
        ranks[start:end] = 0.5 * (start + end - 1) + 1.0
        start = end
    out = np.empty(len(values), dtype=float)
    out[order] = ranks
    return out


def _delong_components(y_true, score_rows):
    """DeLong AUCs and structural components for one or more models.

    Fast mid-rank formulation of Sun & Xu (2014), algebraically identical to
    DeLong, DeLong & Clarke-Pearson (1988).
    """
    y_true = np.asarray(y_true).astype(int)
    scores = np.atleast_2d(np.asarray(score_rows, dtype=float))
    positive = y_true == 1
    negative = ~positive
    m = int(positive.sum())
    n = int(negative.sum())
    if m == 0 or n == 0:
        raise ValueError("AUC requires both classes in y_true")

    k = scores.shape[0]
    tx = np.empty((k, m))
    ty = np.empty((k, n))
    tz = np.empty((k, m + n))
    for row in range(k):
        tx[row] = _midrank(scores[row][positive])
        ty[row] = _midrank(scores[row][negative])
        tz[row] = _midrank(scores[row])

    aucs = (tz[:, positive].sum(axis=1) - m * (m + 1) / 2.0) / (m * n)
    v01 = (tz[:, positive] - tx) / n          # placement values, positives
    v10 = 1.0 - (tz[:, negative] - ty) / m    # placement values, negatives
    return aucs, v01, v10


def delong_single(y_true, scores, alpha=STAT_ALPHA):
    """AUC with an asymptotic DeLong 95% confidence interval (one model)."""
    aucs, v01, v10 = _delong_components(y_true, scores)
    m, n = v01.shape[1], v10.shape[1]
    variance = float(np.var(v01[0], ddof=1) / m + np.var(v10[0], ddof=1) / n)
    z_crit = float(stats.norm.ppf(1.0 - alpha / 2.0))
    half = z_crit * np.sqrt(variance)
    return {
        "auc": float(aucs[0]),
        "ci_low": float(max(0.0, aucs[0] - half)),
        "ci_high": float(min(1.0, aucs[0] + half)),
        "variance": variance,
    }


def delong_paired(y_true, scores_a, scores_b, alpha=STAT_ALPHA):
    """Two-sided DeLong z-test for AUC(B) - AUC(A) on paired predictions."""
    aucs, v01, v10 = _delong_components(y_true, np.vstack([scores_a, scores_b]))
    m, n = v01.shape[1], v10.shape[1]
    covariance = np.cov(v01) / m + np.cov(v10) / n
    delta = float(aucs[1] - aucs[0])
    variance = float(covariance[0, 0] + covariance[1, 1] - 2.0 * covariance[0, 1])
    z_stat = delta / np.sqrt(variance) if variance > 0 else 0.0
    p_value = float(2.0 * stats.norm.sf(abs(z_stat)))
    z_crit = float(stats.norm.ppf(1.0 - alpha / 2.0))
    return {
        "auc_a": float(aucs[0]),
        "auc_b": float(aucs[1]),
        "auc_diff": delta,
        "auc_a_ci": [float(aucs[0] - z_crit * np.sqrt(covariance[0, 0])),
                     float(aucs[0] + z_crit * np.sqrt(covariance[0, 0]))],
        "auc_b_ci": [float(aucs[1] - z_crit * np.sqrt(covariance[1, 1])),
                     float(aucs[1] + z_crit * np.sqrt(covariance[1, 1]))],
        "z": float(z_stat),
        "p_value": p_value,
        "significant": bool(p_value < alpha),
    }


# --- McNemar -----------------------------------------------------------------
def mcnemar_exact(correct_a, correct_b, alpha=STAT_ALPHA):
    """Exact McNemar test plus effect sizes.

    A is the baseline and B the challenger. n01 counts images where A is wrong
    and B is right; n10 counts the reverse. The odds ratio n01/n10 is greater
    than 1 when B is better.
    """
    correct_a = np.asarray(correct_a, dtype=bool)
    correct_b = np.asarray(correct_b, dtype=bool)
    n01 = int(np.sum(~correct_a & correct_b))
    n10 = int(np.sum(correct_a & ~correct_b))
    discordant = n01 + n10

    result = {
        "n01_baseline_wrong_challenger_right": n01,
        "n10_baseline_right_challenger_wrong": n10,
        "discordant": discordant,
        "both_correct": int(np.sum(correct_a & correct_b)),
        "both_wrong": int(np.sum(~correct_a & ~correct_b)),
        "risk_difference": float(correct_b.mean() - correct_a.mean()),
    }
    if discordant == 0:
        result.update({
            "p_exact": 1.0, "chi2_edwards": 0.0, "p_chi2": 1.0,
            "odds_ratio": 1.0, "odds_ratio_ci": [1.0, 1.0],
            "risk_difference_ci": [0.0, 0.0], "significant": False,
        })
        return result

    result["p_exact"] = float(stats.binomtest(min(n01, n10), discordant, 0.5).pvalue)

    chi2 = max((abs(n01 - n10) - 1.0) ** 2 / discordant, 0.0)
    result["chi2_edwards"] = float(chi2)
    result["p_chi2"] = float(stats.chi2.sf(chi2, df=1))

    share = n01 / discordant
    share_low, share_high = clopper_pearson(n01, discordant, alpha)
    result["odds_ratio"] = _odds(share)
    result["odds_ratio_ci"] = [_odds(share_low), _odds(share_high)]

    # Wald interval for the paired risk difference (Agresti, 2002).
    total = len(correct_a)
    standard_error = np.sqrt(max(n01 + n10 - (n01 - n10) ** 2 / total, 0.0)) / total
    z_crit = float(stats.norm.ppf(1.0 - alpha / 2.0))
    diff = (n01 - n10) / total
    result["risk_difference_ci"] = [
        float(max(-1.0, diff - z_crit * standard_error)),
        float(min(1.0, diff + z_crit * standard_error)),
    ]
    result["significant"] = bool(result["p_exact"] < alpha)
    return result


# --- Paired bootstrap --------------------------------------------------------
def paired_bootstrap(y_true, values_a, values_b, metric="accuracy",
                     n_resamples=N_BOOTSTRAP, seed=BOOTSTRAP_SEED,
                     alpha=STAT_ALPHA):
    """Percentile bootstrap CI for metric(B) - metric(A) over paired resamples.

    Both models are evaluated on the same resampled images, which preserves the
    correlation between their per-image performance.
    """
    y_true = np.asarray(y_true)
    a = np.asarray(values_a)
    b = np.asarray(values_b)
    n = len(y_true)

    if metric == "accuracy":
        def score(truth, values):
            return float(np.mean(truth == values))
    elif metric == "auc":
        def score(truth, values):
            return auc_midrank(truth, values)
    else:
        raise ValueError("metric must be 'accuracy' or 'auc'")

    point = score(y_true, b) - score(y_true, a)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_resamples)
    valid = 0
    for _ in range(n_resamples):
        index = rng.integers(0, n, size=n)
        truth = y_true[index]
        try:
            diffs[valid] = score(truth, b[index]) - score(truth, a[index])
        except ValueError:      # a resample containing one class has no AUC
            continue
        valid += 1

    diffs = diffs[:valid]
    low, high = np.percentile(diffs, [100 * alpha / 2.0, 100 * (1 - alpha / 2.0)])
    p_two = min(1.0, 2.0 * min(float(np.mean(diffs <= 0)), float(np.mean(diffs >= 0))))
    return {
        "metric": metric,
        "point_estimate": float(point),
        "ci_low": float(low),
        "ci_high": float(high),
        "p_value": float(p_two),
        "significant": bool(low > 0 or high < 0),
        "n_resamples": int(valid),
        "n_skipped": int(n_resamples - valid),
    }


# --- Multiple-comparison correction -----------------------------------------
def holm_adjust(p_values):
    """Holm step-down adjusted p-values; controls the family-wise error rate."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running_max = 0.0
    for rank, index in enumerate(order):
        running_max = max(running_max, (len(p_values) - rank) * p_values[index])
        adjusted[index] = min(running_max, 1.0)
    return adjusted


# --- Formatting --------------------------------------------------------------
def fmt_p(p):
    return "< 1e-12" if p < 1e-12 else f"{p:.2e}"


def fmt_pct(x, digits=2):
    return f"{100.0 * x:.{digits}f}%"


def fmt_or(x):
    return "inf" if not np.isfinite(x) else f"{x:.2f}"


def jsonable(value):
    """Recursively convert numpy scalars for JSON; non-finite floats -> None."""
    if isinstance(value, dict):
        return {key: jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(item) for item in value]
    if isinstance(value, np.ndarray):
        return jsonable(value.tolist())
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        number = float(value)
        return number if np.isfinite(number) else None
    return value


In [ ]:
print("\n" + "=" * 70)
print("📄 STEP 6.1: PER-IMAGE PREDICTIONS AND INTERVAL ESTIMATES")
print("=" * 70)

# Step 5 evaluated test_loader with shuffle=False, so the arrays in `results`
# are in exactly the same order as test_dataset.files.
y_true = np.asarray(results["targets"]).astype(int)
y_pred = np.asarray(results["predictions"]).astype(int)
y_prob = np.asarray(results["probabilities"], dtype=float)   # [P(NORMAL), P(PNEUMONIA)]
image_paths = list(test_dataset.files)

assert len(image_paths) == len(y_true) == len(y_pred) == len(y_prob), (
    "Evaluation outputs and test images are not aligned. "
    "Re-run Step 5 with test_loader(shuffle=False).")
assert np.allclose(y_prob.sum(axis=1), 1.0, atol=1e-6), "Probabilities must sum to 1."

predictions_df = pd.DataFrame({
    "image_path": image_paths,
    "y_true": y_true,
    "y_pred": y_pred,
    "prob_normal": y_prob[:, 0],
    "prob_pneumonia": y_prob[:, 1],
})
predictions_df.to_csv("test_predictions.csv", index=False)

# Cross-check the export against the metrics printed in Step 5.
assert abs(100.0 * accuracy_score(y_true, y_pred) - results["accuracy"]) < 1e-9
assert abs(roc_auc_score(y_true, y_prob[:, 1]) - results["auc"]) < 1e-9

print(f"✅ test_predictions.csv saved — {len(predictions_df)} rows")
print(f"   NORMAL {int((y_true == 0).sum())} | PNEUMONIA {int((y_true == 1).sum())}")


def ci_text(summary):
    return f"[{fmt_pct(summary['ci_low'])}, {fmt_pct(summary['ci_high'])}]"


vit_summary = model_summary(y_true, y_pred)
vit_auc = delong_single(y_true, y_prob[:, 1])
vit_summary["auc"] = vit_auc["auc"]
vit_summary["auc_ci"] = [vit_auc["ci_low"], vit_auc["ci_high"]]

print("\n📊 ViT-Base — point estimates with 95% intervals")
print("-" * 70)
print(f"   {'Metric':<13}{'Estimate':>10}   {'95% CI':<24}Basis")
print(f"   {'Accuracy':<13}{fmt_pct(vit_summary['accuracy']['estimate']):>10}   "
      f"{ci_text(vit_summary['accuracy']):<24}Wilson")
if "sensitivity" in vit_summary:
    print(f"   {'Sensitivity':<13}{fmt_pct(vit_summary['sensitivity']['estimate']):>10}   "
          f"{ci_text(vit_summary['sensitivity']):<24}Wilson")
if "specificity" in vit_summary:
    print(f"   {'Specificity':<13}{fmt_pct(vit_summary['specificity']['estimate']):>10}   "
          f"{ci_text(vit_summary['specificity']):<24}Wilson")
print(f"   {'AUC':<13}{vit_summary['auc']:>10.3f}   "
      f"[{vit_summary['auc_ci'][0]:.3f}, {vit_summary['auc_ci'][1]:.3f}]{'':<11}DeLong")

counts = vit_summary["confusion"]
print(f"\n   Confusion matrix: TN {counts['tn']} | FP {counts['fp']} | "
      f"FN {counts['fn']} | TP {counts['tp']}")


In [ ]:
import glob
import textwrap

print("\n" + "=" * 70)
print("🧪 STEP 6.2: PAIRED SIGNIFICANCE TESTS AGAINST BASELINES")
print("=" * 70)

correct_vit = (y_pred == y_true)
baseline_records = []
skipped = []

# --- Baseline 1: the trivial majority-class (prevalence) rule ---------------
# Always answering PNEUMONIA requires no training. Beating it is the minimum
# evidence that the ViT has learned more than the class prior.
baseline_records.append({
    "name": "Majority-class rule (always PNEUMONIA)",
    "source": "built-in trivial benchmark",
    "y_pred": np.full_like(y_true, 1),
    "y_prob": None,
})

# --- Baselines 2..k: prediction CSVs found in the environment ---------------
candidate_paths = []
for root in ("/kaggle/input", "/kaggle/working", "."):
    if os.path.isdir(root):
        candidate_paths.extend(
            glob.glob(os.path.join(root, "**", "*predictions*.csv"), recursive=True))

candidate_paths = sorted({os.path.abspath(path) for path in candidate_paths})
candidate_paths = [path for path in candidate_paths
                   if os.path.basename(path) != "test_predictions.csv"]

print(f"🔎 Candidate baseline files (*predictions*.csv): {len(candidate_paths)}")

for path in candidate_paths:
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        skipped.append((os.path.basename(path), f"unreadable ({type(exc).__name__})"))
        continue

    missing = {"image_path", "y_true", "y_pred"} - set(frame.columns)
    if missing:
        skipped.append((os.path.basename(path),
                        f"missing column(s): {', '.join(sorted(missing))}"))
        continue
    if frame["image_path"].duplicated().any():
        skipped.append((os.path.basename(path), "duplicate image_path rows"))
        continue
    if set(frame["image_path"]) != set(image_paths):
        skipped.append((os.path.basename(path),
                        f"different test set ({len(frame)} vs {len(image_paths)} images)"))
        continue

    frame = frame.set_index("image_path").loc[image_paths]
    if not np.array_equal(frame["y_true"].to_numpy().astype(int), y_true):
        skipped.append((os.path.basename(path), "labels do not match the ViT export"))
        continue

    baseline_records.append({
        "name": os.path.basename(path),
        "source": path,
        "y_pred": frame["y_pred"].to_numpy().astype(int),
        "y_prob": (frame["prob_pneumonia"].to_numpy(dtype=float)
                   if "prob_pneumonia" in frame.columns else None),
    })

for name, reason in skipped:
    print(f"   ⏭️  skipped {name}: {reason}")

# --- Run every test for every baseline --------------------------------------
for record in baseline_records:
    pred_a = np.asarray(record["y_pred"]).astype(int)
    record["summary"] = model_summary(y_true, pred_a)
    record["mcnemar"] = mcnemar_exact(pred_a == y_true, correct_vit, alpha=STAT_ALPHA)
    record["bootstrap_accuracy"] = paired_bootstrap(
        y_true, pred_a, y_pred, metric="accuracy")
    record["delong"] = None
    record["bootstrap_auc"] = None
    if record["y_prob"] is not None:
        prob_a = np.asarray(record["y_prob"], dtype=float)
        record["delong"] = delong_paired(
            y_true, prob_a, y_prob[:, 1], alpha=STAT_ALPHA)
        record["bootstrap_auc"] = paired_bootstrap(
            y_true, prob_a, y_prob[:, 1], metric="auc")

# --- Holm-Bonferroni across the family of baselines -------------------------
holm_mcnemar = holm_adjust(np.array([r["mcnemar"]["p_exact"] for r in baseline_records]))
for record, adjusted in zip(baseline_records, holm_mcnemar):
    record["mcnemar"]["p_holm"] = float(adjusted)

with_auc = [r for r in baseline_records if r["delong"] is not None]
if with_auc:
    holm_delong = holm_adjust(np.array([r["delong"]["p_value"] for r in with_auc]))
    for record, adjusted in zip(with_auc, holm_delong):
        record["delong"]["p_holm"] = float(adjusted)

# --- Summary table ----------------------------------------------------------
print(f"\n📋 Family of {len(baseline_records)} comparison(s); "
      f"Holm–Bonferroni across baselines (alpha = {STAT_ALPHA})")
print("-" * 104)
print(f"{'Baseline':40s} {'Δacc(pp)':>9s} {'McNemar p':>10s} {'Holm p':>10s} "
      f"{'ΔAUC':>8s} {'DeLong p':>9s} {'Holm p':>9s}  Verdict")
print("-" * 104)
for record in baseline_records:
    mcnemar = record["mcnemar"]
    delong = record["delong"]
    delong_p = fmt_p(delong["p_value"]) if delong else "n/a"
    delong_holm = fmt_p(delong["p_holm"]) if delong else "n/a"
    auc_diff = f"{delong['auc_diff']:+.3f}" if delong else "n/a"
    accuracy_ok = mcnemar["p_holm"] < STAT_ALPHA
    auc_ok = (delong is None) or (delong["p_holm"] < STAT_ALPHA)
    verdict = "YES" if (accuracy_ok and auc_ok) else ("partial" if (accuracy_ok or auc_ok) else "no")
    print(f"{record['name'][:40]:40s} "
          f"{100 * record['bootstrap_accuracy']['point_estimate']:>+8.2f} "
          f"{fmt_p(mcnemar['p_exact']):>10s} {fmt_p(mcnemar['p_holm']):>10s} "
          f"{auc_diff:>8s} {delong_p:>9s} {delong_holm:>9s}  {verdict}")
print("-" * 104)
print("Verdict = significant for accuracy AND (where available) AUC after Holm correction.")

# --- Detail for each baseline ----------------------------------------------
for record in baseline_records:
    mcnemar = record["mcnemar"]
    delong = record["delong"]
    bootstrap = record["bootstrap_accuracy"]
    print(f"\n🔬 ViT vs {record['name']}")
    print(f"   source: {record['source']}")
    print(f"   accuracy: baseline {fmt_pct(record['summary']['accuracy']['estimate'])}, "
          f"ViT {fmt_pct(vit_summary['accuracy']['estimate'])}, "
          f"Δ {100 * bootstrap['point_estimate']:+.2f} pp "
          f"(bootstrap 95% CI {100 * bootstrap['ci_low']:+.2f} to "
          f"{100 * bootstrap['ci_high']:+.2f})")
    print(f"   McNemar: discordant {mcnemar['discordant']} "
          f"(baseline wrong/ViT right {mcnemar['n01_baseline_wrong_challenger_right']}, "
          f"baseline right/ViT wrong {mcnemar['n10_baseline_right_challenger_wrong']})")
    print(f"            exact p = {fmt_p(mcnemar['p_exact'])} | "
          f"Edwards χ² p = {fmt_p(mcnemar['p_chi2'])}")
    print(f"            OR = {fmt_or(mcnemar['odds_ratio'])} "
          f"(95% exact CI {fmt_or(mcnemar['odds_ratio_ci'][0])} to "
          f"{fmt_or(mcnemar['odds_ratio_ci'][1])}) | "
          f"risk difference {100 * mcnemar['risk_difference']:+.2f} pp "
          f"(95% CI {100 * mcnemar['risk_difference_ci'][0]:+.2f} to "
          f"{100 * mcnemar['risk_difference_ci'][1]:+.2f})")
    if delong:
        print(f"   DeLong:  AUC {delong['auc_a']:.4f} → {delong['auc_b']:.4f} "
              f"(Δ {delong['auc_diff']:+.4f}), z = {delong['z']:.3f}, "
              f"p = {fmt_p(delong['p_value'])}, Holm p = {fmt_p(delong['p_holm'])}")
        bootstrap_auc = record["bootstrap_auc"]
        print(f"            bootstrap 95% CI for ΔAUC "
              f"[{bootstrap_auc['ci_low']:+.4f}, {bootstrap_auc['ci_high']:+.4f}]")
    else:
        print("   DeLong:  not applicable (constant scores, so AUC is undefined)")
    print(f"   Holm-adjusted McNemar p = {fmt_p(mcnemar['p_holm'])} "
          f"→ {'significant' if mcnemar['p_holm'] < STAT_ALPHA else 'not significant'}")

# --- Publication-ready sentences -------------------------------------------
sentences = []
accuracy = vit_summary["accuracy"]
sentences.append(
    f"The Vision Transformer achieved {fmt_pct(accuracy['estimate'])} test accuracy "
    f"(95% Wilson CI {fmt_pct(accuracy['ci_low'])} to {fmt_pct(accuracy['ci_high'])}, "
    f"n = {accuracy['n']}).")

if "sensitivity" in vit_summary and "specificity" in vit_summary:
    sensitivity = vit_summary["sensitivity"]
    specificity = vit_summary["specificity"]
    sentences.append(
        f"Sensitivity was {fmt_pct(sensitivity['estimate'])} "
        f"(95% CI {fmt_pct(sensitivity['ci_low'])} to {fmt_pct(sensitivity['ci_high'])}) "
        f"and specificity {fmt_pct(specificity['estimate'])} "
        f"(95% CI {fmt_pct(specificity['ci_low'])} to {fmt_pct(specificity['ci_high'])}).")

sentences.append(
    f"Discrimination reached AUC {vit_summary['auc']:.3f} "
    f"(95% DeLong CI {vit_summary['auc_ci'][0]:.3f} to {vit_summary['auc_ci'][1]:.3f}).")

for record in baseline_records:
    mcnemar = record["mcnemar"]
    bootstrap = record["bootstrap_accuracy"]
    direction = "higher" if bootstrap["point_estimate"] >= 0 else "lower"
    sentences.append(
        f"Relative to {record['name']}, ViT accuracy was "
        f"{abs(100 * bootstrap['point_estimate']):.2f} percentage points {direction} "
        f"(paired bootstrap 95% CI {100 * bootstrap['ci_low']:+.2f} to "
        f"{100 * bootstrap['ci_high']:+.2f} pp); exact McNemar p = "
        f"{fmt_p(mcnemar['p_exact'])}, Holm-adjusted p = {fmt_p(mcnemar['p_holm'])}, "
        f"odds ratio {fmt_or(mcnemar['odds_ratio'])} (95% exact CI "
        f"{fmt_or(mcnemar['odds_ratio_ci'][0])} to "
        f"{fmt_or(mcnemar['odds_ratio_ci'][1])}).")
    if record["delong"]:
        delong = record["delong"]
        sentences.append(
            f"The paired AUC difference was {delong['auc_diff']:+.3f} "
            f"(DeLong z = {delong['z']:.2f}, p = {fmt_p(delong['p_value'])}, "
            f"Holm-adjusted p = {fmt_p(delong['p_holm'])}).")

print("\n📝 Publication-ready sentences")
print("-" * 70)
for sentence in sentences:
    print(textwrap.fill(sentence, width=88))
    print()

# --- Persist every statistic -------------------------------------------------
statistical_results = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "settings": {
        "alpha": STAT_ALPHA,
        "n_bootstrap": N_BOOTSTRAP,
        "seed": BOOTSTRAP_SEED,
        "n_test_images": int(len(y_true)),
        "class_balance": {
            "NORMAL": int((y_true == 0).sum()),
            "PNEUMONIA": int((y_true == 1).sum()),
        },
    },
    "reference_model": {
        "name": "ViT-Base/16",
        "summary": vit_summary,
    },
    "baselines": [
        {
            "name": record["name"],
            "source": record["source"],
            "summary": record["summary"],
            "mcnemar": record["mcnemar"],
            "bootstrap_accuracy": record["bootstrap_accuracy"],
            "delong": record["delong"],
            "bootstrap_auc": record["bootstrap_auc"],
        }
        for record in baseline_records
    ],
    "sentences": sentences,
}
with open("statistical_results.json", "w") as handle:
    json.dump(jsonable(statistical_results), handle, indent=2)

print("📦 Saved statistical_results.json")


In [ ]:
print("\n" + "="*70)
print("💾 STEP 7: MODEL SAVING AND EXPORT FOR DOWNLOAD")
print("="*70)

def save_kaggle_outputs(model, history, results):
    """Save all outputs for Kaggle download"""
    print("💾 Saving model and artifacts...")
    
    # Model files
    torch.save(model.state_dict(), 'pneumonia_vit_model.pth')
    torch.save(model, 'pneumonia_vit_complete.pth')
    print("✅ Model files saved")
    
    # Training data
    np.save('training_history.npy', history)
    np.save('evaluation_results.npy', results)
    print("✅ Training data saved")
    
    # Model info
    model_info = {
        'model_name': MODEL_NAME,
        'accuracy': results['accuracy'],
        'auc': results['auc'],
        'num_parameters': sum(p.numel() for p in model.parameters()),
        'class_names': ['NORMAL', 'PNEUMONIA'],
        'input_size': INPUT_SIZE,
        'normalization': {'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]}
    }
    np.save('model_info.npy', model_info)
    print("✅ Model info saved")
    
    # Summary report
    with open('training_summary.txt', 'w') as f:
        f.write("🫁 PNEUMONIA DETECTION - KAGGLE TRAINING RESULTS\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Model: {MODEL_NAME}\n")
        f.write(f"Best Validation Accuracy: {max(history['val_acc']):.2f}%\n")
        f.write(f"Test Accuracy: {results['accuracy']:.2f}%\n")
        f.write(f"AUC Score: {results['auc']:.3f}\n")
        f.write(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}\n")
        f.write(f"Training Epochs: {NUM_EPOCHS}\n")
        f.write(f"Batch Size: {BATCH_SIZE}\n")
        f.write(f"Quality Control: {REMOVE_PROBLEMATIC}\n")
    
    print("✅ Summary report saved")
    
    # List all output files
    files = [f for f in os.listdir('.') if f.endswith(('.pth', '.npy', '.txt', '.csv', '.json'))]
    total_size = sum(os.path.getsize(f) for f in files) / (1024**2)
    
    print(f"\n📦 FILES READY FOR DOWNLOAD:")
    for f in sorted(files):
        size = os.path.getsize(f) / (1024**2)
        print(f"   📄 {f} ({size:.1f} MB)")
    print(f"   📊 Total: {total_size:.1f} MB")

save_kaggle_outputs(trained_model, training_history, results)

In [ ]:
print("\n" + "="*70)
print("🔬 STEP 8: SAMPLE INFERENCE AND VISUALIZATION")
print("="*70)

def demonstrate_inference(model, test_dataset, num_samples=8):
    """Demonstrate model inference on sample images"""
    model.eval()
    class_names = ['NORMAL', 'PNEUMONIA']
    
    # Get random samples
    indices = np.random.choice(len(test_dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('🔬 Sample Predictions', fontsize=16, fontweight='bold')
    
    for i, idx in enumerate(indices):
        image, true_label = test_dataset[idx]
        
        # Make prediction
        with torch.no_grad():
            input_tensor = image.unsqueeze(0).to(device)
            output = model(input_tensor)
            prob = F.softmax(output, dim=1)
            pred_label = output.argmax(dim=1).item()
            confidence = prob.max().item()
        
        # Display image
        row, col = i // 4, i % 4
        ax = axes[row, col]
        
        # Denormalize for display
        img_display = image * 0.5 + 0.5
        img_display = img_display.permute(1, 2, 0).cpu().numpy()
        
        ax.imshow(img_display, cmap='gray')
        ax.set_title(f'True: {class_names[true_label]}\n'
                    f'Pred: {class_names[pred_label]} ({confidence:.1%})')
        ax.axis('off')
        
        # Color border based on correctness
        color = 'green' if pred_label == true_label else 'red'
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
    
    plt.tight_layout()
    plt.show()

# Demonstrate inference
demonstrate_inference(trained_model, test_dataset)

In [ ]:
print("\n" + "="*70)
print("🎉 KAGGLE PNEUMONIA DETECTION PIPELINE COMPLETED!")
print("="*70)

best_val_acc = max(training_history['val_acc'])
test_acc = results['accuracy']
auc_score = results['auc']
n_significant = sum(1 for record in baseline_records
                    if record['mcnemar']['p_holm'] < STAT_ALPHA)

print(f"""
🏆 FINAL RESULTS:

📊 Dataset:
   • Total images processed: {len(quality_df):,}
   • Clean training images: {len(train_dataset):,}
   • Quality control applied: ✅

🤖 Model Performance:
   • Architecture: Vision Transformer (ViT-Base)
   • Best validation accuracy: {best_val_acc:.2f}%
   • Test accuracy: {test_acc:.2f}%
   • AUC score: {auc_score:.3f}
   • Parameters: {sum(p.numel() for p in trained_model.parameters()):,}

📐 Statistical rigour:
   • Baselines compared: {len(baseline_records)}
   • Significant after Holm correction: {n_significant}
   • Per-image predictions: test_predictions.csv
   • Full statistics: statistical_results.json

⚙️ Training:
   • Epochs: {NUM_EPOCHS}
   • Device: {device}
   • Enhanced quality control: ✅
   • Medical-grade preprocessing: ✅

💾 Output Files (ready for download):
   • pneumonia_vit_model.pth (model weights)
   • pneumonia_vit_complete.pth (complete model)
   • training_history.npy (training metrics)
   • evaluation_results.npy (test results)
   • model_info.npy (model metadata)
   • training_summary.txt (detailed report)
   • test_predictions.csv (per-image probabilities)
   • statistical_results.json (paired test results)

🚀 Next Steps:
   1. Download all .pth and .npy files
   2. Use for local inference and deployment
   3. Report the paired tests in the dissertation's results chapter
   4. Deploy with Gradio/Streamlit web interface

⚠️ Medical Disclaimer:
This AI model is for educational purposes only.
Always consult healthcare professionals for medical decisions.
""")

print("="*70)
print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("🎯 Your enhanced pneumonia detection model is ready!")
print("="*70)

# Memory cleanup
free_gpu_cache()
gc.collect()

In [ ]:
# =============================================================================
# APPENDIX (OPTIONAL): SALIENCY MAP WITH A PRETRAINED DENSENET121
# =============================================================================
# Illustrative only: this uses an ImageNet DenseNet121, not the ViT trained
# above, and downloads one example image. It is skipped gracefully when
# TensorFlow or the download is unavailable.

try:
    import tensorflow as tf
    from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
    from tensorflow.keras.preprocessing import image as keras_image

    print("\n" + "=" * 70)
    print("🖼️ APPENDIX: SALIENCY MAP WITH PRETRAINED DENSENET121")
    print("=" * 70)

    img_url = "https://github.com/ieee8023/covid-chestxray-dataset/raw/master/images/000003-2.jpg"
    img_path = tf.keras.utils.get_file("demo_cxr_new.jpg", img_url)
    img = keras_image.load_img(img_path, target_size=(224, 224))
    img_array = keras_image.img_to_array(img)
    x_input = preprocess_input(np.expand_dims(img_array, axis=0).copy())

    saliency_model = DenseNet121(weights="imagenet")
    preds = saliency_model.predict(x_input, verbose=0)
    class_idx = int(np.argmax(preds[0]))

    with tf.GradientTape() as tape:
        x_tensor = tf.convert_to_tensor(x_input)
        tape.watch(x_tensor)
        pred_score = saliency_model(x_tensor)[0][class_idx]
    grads = tape.gradient(pred_score, x_tensor)[0].numpy()

    saliency = np.max(np.abs(grads), axis=-1)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(12, 6))
    axes[0].imshow(img)
    axes[0].set_title("Original CXR")
    axes[0].axis("off")
    axes[1].imshow(saliency, cmap="jet")
    axes[1].set_title("Saliency map")
    axes[1].axis("off")
    axes[2].imshow(img)
    axes[2].imshow(saliency, cmap="jet", alpha=0.5)
    axes[2].set_title("Overlay")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

    print("Predicted ImageNet class index:", class_idx)
    print(f"Prediction confidence: {preds[0][class_idx]:.3f}")
except Exception as exc:
    print(f"⚠️ Appendix skipped: {type(exc).__name__}: {exc}")
